# Experiment 2: NN nuisance with \(\theta\) fixed at \(\theta_0\)

Zhong–Wang Simulation I, Case 1, homoscedastic \(t_3\), \(\tau=0.5\).

**Step 1.** Train a dense ReLU MLP for \(m\) with \(\theta\equiv\theta_0\) (not trainable). Exactly `EPOCHS` epochs — **no early stopping**.

**Step 2.** Freeze \(\hat m\), then estimate \(\theta\) by global convex median regression (`scipy.optimize.linprog`).

Oracle \(f_0\) and \(\varphi^*\) throughout. Known \(m_0\) only for diagnostics.

You can either run the cells below or execute `run_experiment2.py` from this folder. Results append under `run/` and skip completed replications.


## Why \(m^*=m_0\)

On \(z_j\in[0,2]\), \(\mathrm{ReLU}(z_j)=z_j\). Hence \(m_0(z)=0.56\sum_{j=1}^8 z_j\) can be represented exactly by a ReLU net with identity paths and final weights \(0.56\). We use widths `[32,32,16]` with **no sparsity mask**, so \(m_0\) lies in the network class and we may take \(m^*=m_0\). Observed nuisance error is finite-sample estimation / NN optimization error only.


## Interpretation vs Experiment 0

Experiment 2 adds NN nuisance estimation but **keeps \(\theta\) at truth while \(m\) trains**, then re-estimates \(\theta\) by a global convex QR solve. There is no channel \(\theta\)-error \(\to\) nuisance-training error. Differences from Experiment 0 are attributable primarily to finite-sample nuisance fitting and its empirical interaction with the score — not joint \(\theta\)–\(m\) feedback. \(f\) and \(\varphi^*\) remain oracle.


In [1]:
import importlib.util
from pathlib import Path

_path = Path("run_experiment2.py").resolve()
_spec = importlib.util.spec_from_file_location("exp2", _path)
exp2 = importlib.util.module_from_spec(_spec)
# Load module without executing __main__ block — __name__ will be "exp2"
_spec.loader.exec_module(exp2)

globals().update({k: getattr(exp2, k) for k in dir(exp2) if not k.startswith("_")})
print("F0 =", F0)
print("HIDDEN =", HIDDEN, "EPOCHS =", EPOCHS, "(no early stopping)")
print("cwd =", ROOT)


F0 = 0.3675525969478614
HIDDEN = [32, 32, 16] EPOCHS = 1000 (no early stopping)
cwd = C:\Users\Tiansui Tu\Documents\GitHub\dplqr\results\2026-09-21_case1_oracle_diagnostics\experiment2_fixed_theta_nn


## Write config only (safe; does not run Monte Carlo)


In [2]:
config = write_config()
print("Wrote", CONFIG_JSON)
print(json.dumps(config, indent=2))


Wrote C:\Users\Tiansui Tu\Documents\GitHub\dplqr\results\2026-09-21_case1_oracle_diagnostics\experiment2_fixed_theta_nn\run\config.json
{
  "experiment": "experiment2_fixed_theta_nn",
  "N_VALUES": [
    1000,
    2000
  ],
  "Q": 50,
  "BASE_SEED": 20260921,
  "TAU": 0.5,
  "THETA0": [
    1.0,
    -1.0
  ],
  "M0_COEF": 0.56,
  "DF_T": 3,
  "F0": 0.3675525969478614,
  "WALD_Z": 1.959963984540054,
  "HIDDEN": [
    32,
    32,
    16
  ],
  "EPOCHS": 1000,
  "LEARNING_RATE": 0.001,
  "BATCH_SIZE": 128,
  "early_stopping": false,
  "sparsity_mask": false,
  "theta_during_m_train": "fixed_at_theta0",
  "theta_solver": "scipy.optimize.linprog(method='highs') L1 / median LP",
  "device": "cpu",
  "note": "NN m with theta glued at truth; then convex theta; oracle f0 and varphi*."
}


## Monte Carlo loop (run manually)

Restartable. **No early stopping** inside `train_m_fixed_theta` — always trains exactly `EPOCHS` epochs.


In [3]:
completed = load_completed()
print(f"Already completed: {len(completed)} rows")

for n in N_VALUES:
    for rep in range(1, Q + 1):
        key = (n, rep)
        if key in completed:
            continue
        row = run_one(n, rep)
        append_result(row)
        completed.add(key)
        if rep % 10 == 0 or rep == 1:
            print(
                f"n={n} rep={rep}/{Q}  "
                f"theta_hat={row['theta1_hat']:.4f},{row['theta2_hat']:.4f}  "
                f"nuisance_L2={row['nuisance_L2']:.4f}  "
                f"Cn_norm={row['Cn_norm']:.3f}  "
                f"IF_gap_norm={row['IF_gap_norm']:.3f}"
            )

print("Done. Results at", RESULTS_CSV)


Already completed: 8 rows
n=1000 rep=10/50  theta_hat=1.0156,-0.9903  nuisance_L2=0.5325  Cn_norm=0.161  IF_gap_norm=0.167
n=1000 rep=20/50  theta_hat=0.9396,-1.0454  nuisance_L2=0.4857  Cn_norm=0.163  IF_gap_norm=0.109
n=1000 rep=30/50  theta_hat=0.9879,-1.0570  nuisance_L2=0.6380  Cn_norm=0.074  IF_gap_norm=0.209
n=1000 rep=40/50  theta_hat=0.9586,-0.9693  nuisance_L2=0.5791  Cn_norm=0.110  IF_gap_norm=0.167
n=1000 rep=50/50  theta_hat=0.9996,-1.0044  nuisance_L2=0.6155  Cn_norm=0.039  IF_gap_norm=0.169
n=2000 rep=1/50  theta_hat=0.9754,-1.1329  nuisance_L2=0.4606  Cn_norm=0.123  IF_gap_norm=0.538
n=2000 rep=10/50  theta_hat=0.9849,-1.1100  nuisance_L2=0.4249  Cn_norm=0.096  IF_gap_norm=0.538
n=2000 rep=20/50  theta_hat=1.0548,-0.9788  nuisance_L2=0.5189  Cn_norm=0.088  IF_gap_norm=0.115
n=2000 rep=30/50  theta_hat=1.0592,-1.0366  nuisance_L2=0.5729  Cn_norm=0.076  IF_gap_norm=0.260
n=2000 rep=40/50  theta_hat=0.9830,-0.9812  nuisance_L2=0.5213  Cn_norm=0.108  IF_gap_norm=0.153
n=200

## Summary table


In [4]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError("Run the Monte Carlo cell first to create replication_results.csv")

df = pd.read_csv(RESULTS_CSV)
summary = summarize(df)
summary.to_csv(SUMMARY_CSV, index=False)
print(summary.to_string(index=False))
print("Wrote", SUMMARY_CSV)


   n  Q  theta1_mean  theta1_bias  theta1_sqrtn_bias  theta1_mc_sd  theta1_mean_oracle_se   T1_mean    T1_sd  coverage1  theta2_mean  theta2_bias  theta2_sqrtn_bias  theta2_mc_sd  theta2_mean_oracle_se   T2_mean    T2_sd  coverage2  nuisance_L2_mean  nuisance_L2_sd  Cn1_mean   Cn1_sd  Cn2_mean   Cn2_sd  Cn_norm_mean  IF_gap1_mean  IF_gap2_mean  IF_gap_norm_mean  final_training_loss_mean
1000 51     0.989238    -0.010762          -0.340334      0.044511               0.102898 -0.104088 0.432638        1.0    -1.013424    -0.013424          -0.424504      0.049304               0.098954 -0.135753 0.500043       1.00          0.555148        0.062584 -0.011001 0.103593  0.023766 0.088189       0.12344      0.026707     -0.043635          0.257551                  0.472239
2000 50     0.995959    -0.004041          -0.180705      0.044845               0.072663 -0.056365 0.619255        1.0    -1.006130    -0.006130          -0.274160      0.051808               0.069765 -0.087964 0.742352

## Figures


In [5]:
df = pd.read_csv(RESULTS_CSV)
make_figures(df)
print("Figures written to", FIG_DIR)
for p in sorted(FIG_DIR.glob("*.png")):
    print(" ", p.name)


Figures written to C:\Users\Tiansui Tu\Documents\GitHub\dplqr\results\2026-09-21_case1_oracle_diagnostics\experiment2_fixed_theta_nn\run\figures
  Cn_hist_n1000.png
  Cn_hist_n2000.png
  Cn_norm_boxplot.png
  Cn_norm_hist_n1000.png
  Cn_norm_hist_n2000.png
  IF_scatter_n1000.png
  IF_scatter_n2000.png
  nuisance_L2_boxplot.png
  nuisance_L2_hist_n1000.png
  nuisance_L2_hist_n2000.png
  T1_hist_n1000.png
  T1_hist_n2000.png
  T1_qq_n1000.png
  T1_qq_n2000.png
  T2_qq_n1000.png
  T2_qq_n2000.png
